# Advanced Simulation of Power System Stability and Cascading Failure Dynamics in Resilient Smart Grids

**Module:** Advanced Simulation Techniques (AST) &nbsp;|&nbsp; **Task 2 — Simulation Design and Implementation**  
**Author:** Salina Khadka, BSBI Berlin  
**Tool:** Python + SimPy (discrete-event simulation)

---

This notebook is the interactive Google Colab front-end for the discrete-event smart-grid model. It lets you:

1. **Control simulation parameters in real time** (sliders for demand, failure probability, repair crews, etc.).
2. **View results** through line plots, a failure timeline and a per-line load **heatmap**.
3. **Export results** to CSV for further analysis.
4. Run the **what-if scenario comparison** and a **multi-seed accuracy study**.

The simulation logic lives in separate, version-controlled `.py` files (`config.py`, `grid_model.py`, `metrics.py`, `scenarios.py`, `visualization.py`) so this notebook stays a thin user interface. Run the **Setup** cell first.

## 1. Setup — install dependencies and load the model files

When this notebook is opened from GitHub in Colab, the supporting `.py` files are not present, so this cell clones the repository and changes into it. Run it once.

In [ ]:
import os, sys, subprocess

# Install SimPy (matplotlib / numpy / ipywidgets are pre-installed in Colab).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "simpy"], check=False)

REPO_URL = "https://github.com/khadkasaleena/smart-grid-simulation.git"
REPO_DIR = "smart-grid-simulation"

# If the model files are not already next to the notebook, clone the repo.
if not os.path.exists("grid_model.py"):
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    if os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Model files present:", sorted(f for f in os.listdir('.') if f.endswith('.py')))

In [ ]:
# Imports used throughout the notebook.
import importlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import config, grid_model, metrics, scenarios, visualization
# Reload in case the cell is re-run after editing a file.
for m in (config, grid_model, metrics, scenarios, visualization):
    importlib.reload(m)

from grid_model import GridSimulation
from metrics import summarise, accuracy_study
from scenarios import run_all
from visualization import (plot_power_output, plot_grid_load, plot_failures,
                           plot_line_heatmap, export_timeseries_csv,
                           export_event_log_csv)
print("Imports OK. Installed capacity:", config.TOTAL_CAPACITY, "MW")

## 2. Conceptual system diagram (Task 2.1)

Generation → transmission lines → substations → consumer zones, with the repair-crew control loop closing back onto failed lines.

In [ ]:
import make_system_diagram
make_system_diagram.main()
from IPython.display import Image
Image("system_diagram.png", width=900)

## 3. Interactive control panel (Task 2.3 — real-time parameter control + visualisation)

Move the sliders to change the grid configuration, then click **Run simulation**. The four visualisations (generation vs demand, average line load, failure timeline, and the per-line load heatmap) redraw inline, and the headline metrics are printed below.

In [ ]:
# --- Widgets -------------------------------------------------------------
style = {"description_width": "170px"}
layout = widgets.Layout(width="480px")

w_duration   = widgets.IntSlider(value=72, min=24, max=240, step=12,
                                 description="Sim duration (h)", style=style, layout=layout)
w_demand     = widgets.FloatSlider(value=1.0, min=0.6, max=1.6, step=0.05,
                                   description="Demand multiplier", style=style, layout=layout)
w_failprob   = widgets.FloatSlider(value=0.05, min=0.0, max=0.30, step=0.01,
                                   description="Base failure prob/h", style=style, layout=layout, readout_format=".2f")
w_crews      = widgets.IntSlider(value=1, min=1, max=5, step=1,
                                 description="Repair crews", style=style, layout=layout)
w_solar      = widgets.FloatSlider(value=1.0, min=0.5, max=1.6, step=0.05,
                                   description="Solar scale", style=style, layout=layout)
w_cascade    = widgets.FloatSlider(value=1.20, min=1.0, max=1.6, step=0.05,
                                   description="Cascade threshold", style=style, layout=layout, readout_format=".2f")
w_seed       = widgets.IntText(value=42, description="Random seed", style=style, layout=layout)

run_btn = widgets.Button(description="▶ Run simulation", button_style="success",
                         layout=widgets.Layout(width="200px"))
export_btn = widgets.Button(description="⬇ Export CSV", button_style="info",
                            layout=widgets.Layout(width="200px"))
out = widgets.Output()

# Hold the most recent finished simulation so the Export button can use it.
state = {"sim": None}

def build_config():
    return {
        "sim_duration":   int(w_duration.value),
        "demand_multiplier": float(w_demand.value),
        "base_fail_prob": float(w_failprob.value),
        "num_repair_crews": int(w_crews.value),
        "solar_scale":    float(w_solar.value),
        "cascade_thresh": float(w_cascade.value),
    }

def on_run(_):
    with out:
        clear_output(wait=True)
        sim = GridSimulation(config=build_config(), seed=int(w_seed.value))
        results = sim.run()
        state["sim"] = sim
        print("Headline metrics")
        print("-" * 40)
        for k, v in summarise(results).items():
            print(f"  {k:18s}: {v}")
        for builder in (plot_power_output, plot_grid_load, plot_failures, plot_line_heatmap):
            fig = builder(sim)
            plt.show()
            plt.close(fig)

def on_export(_):
    with out:
        if state["sim"] is None:
            print("Run a simulation first."); return
        ts = export_timeseries_csv(state["sim"], "simulation_results.csv")
        ev = export_event_log_csv(state["sim"], "event_log.csv")
        print(f"Exported {ts} and {ev}.")
        try:
            from google.colab import files
            files.download(ts); files.download(ev)
        except Exception:
            print("(Download only works inside Google Colab; files saved to disk.)")

run_btn.on_click(on_run)
export_btn.on_click(on_export)

controls = widgets.VBox([w_duration, w_demand, w_failprob, w_crews, w_solar, w_cascade, w_seed,
                         widgets.HBox([run_btn, export_btn])])
display(controls, out)
on_run(None)  # run once with defaults so the panel is populated

## 4. What-if scenario comparison (Task 2.4)

Runs all six predefined scenarios through the same model and tabulates the headline resilience metrics.

In [ ]:
import pandas as pd
rows = run_all(seed=42)
df = pd.DataFrame(rows)[["scenario", "failures", "cascade_events", "mttr_h",
                          "peak_load_pct", "crew_utilisation", "unrecovered", "exec_ms"]]
display(df)
df.to_csv("scenario_comparison.csv", index=False)
print("Saved scenario_comparison.csv")

In [ ]:
# Bar chart of failures and cascade events per scenario.
ax = df.set_index("scenario")[["failures", "cascade_events"]].plot.bar(
    figsize=(11, 5), color=["#2980b9", "#e74c3c"])
ax.set_ylabel("Count over simulation horizon")
ax.set_title("Failures and Cascade Events by Scenario", fontweight="bold")
plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()

## 5. Accuracy study (Task 2.4 — accuracy measures)

Because the model is stochastic, a single run is not enough. We repeat the baseline across 30 random seeds and report the mean and 95% confidence half-width for each metric — this confidence interval is our accuracy measure.

In [ ]:
report = accuracy_study(n_runs=30)
acc = pd.DataFrame([
    {"metric": k, "mean": round(v["mean"], 3), "std": round(v["std"], 3),
     "ci95_halfwidth": round(v["ci95"], 3), "n": v["n"]}
    for k, v in report.items()
])
display(acc)

---
*End of notebook. The full written report (Tasks 1–4) accompanies this submission as a separate document. Source files are included in this repository as the Task 2 appendix.*